In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

DATA_DIR = '../data'

transactions = pd.read_csv(f'{DATA_DIR}/transactions.csv', parse_dates=['timestamp'])
accounts = pd.read_csv(f'{DATA_DIR}/accounts.csv')
customers = pd.read_csv(f'{DATA_DIR}/customers.csv', parse_dates=['join_date'])
counterparties = pd.read_csv(f'{DATA_DIR}/counterparties.csv')
ground_truth = pd.read_csv(f'{DATA_DIR}/ground_truth.csv')
risk_profiles = pd.read_csv(f'{DATA_DIR}/customer_risk_profiles.csv')

First I'm isolating cash transactions

In [3]:
cash = transactions[transactions['transaction_type'] == 'Cash Deposit'].copy()
cash = cash[cash['direction'] == 'Credit']
cash = cash.sort_values(['customer_id', 'timestamp']).reset_index(drop=True)

print(f"Total cash deposits: {len(cash):,}")
print(f"Unique customers with cash deposits: {cash['customer_id'].nunique():,}")
cash['amount_aed_equivalent'].describe()

Total cash deposits: 726
Unique customers with cash deposits: 149


count      726.000000
mean     18527.109601
std      13112.617463
min       7158.430000
25%      11441.805000
50%      13764.695000
75%      17262.045000
max      76747.160000
Name: amount_aed_equivalent, dtype: float64

There is a caveat in this, and I'll discuss about that in the validation part. But don't worry, the detection rules are on point with industry standards and nothing to be concerned about that

There are 3 types of smurfing scenario we are going to discuss. starting with 1 Day Rapid Smurfing.

**1 day Rapid smurfing**

In [4]:
cash = cash[cash['amount_aed_equivalent'] < 55000].copy()
cash['timestamp'] = pd.to_datetime(cash['timestamp'])

scenario1_alerts = []

for cust_id, group in cash.groupby('customer_id'):
    group = group.sort_values('timestamp').reset_index(drop=True)
    
    for i in range(len(group)):
        window_start = group.loc[i, 'timestamp']
        window_end = window_start + pd.Timedelta(days=1)
        
        window_txns = group[(group['timestamp'] >= window_start) & (group['timestamp'] < window_end)]
        
        count = len(window_txns)
        total = window_txns['amount_aed_equivalent'].sum()
        
        if count >= 2 and total >= 50000:
            scenario1_alerts.append({
                'customer_id': cust_id,
                'scenario': '1-Day Rapid Smurfing',
                'window_start': window_start,
                'window_end': window_end,
                'num_deposits': count,
                'total_amount': round(total, 2),
                'transaction_ids': list(window_txns['transaction_id'])
            })

scenario1_df = pd.DataFrame(scenario1_alerts)
scenario1_df = scenario1_df.drop_duplicates(subset=['customer_id', 'window_start'])

print(f"Scenario 1 alerts: {len(scenario1_df)}")
scenario1_df.head()

Scenario 1 alerts: 28


,customer_id,scenario,window_start,window_end,num_deposits,total_amount,transaction_ids
0,CUST_102615,1-Day Rapid Smurfing,2023-03-11 14:00:00,2023-03-12 14:00:00,2,92854.57,"[TXN_AACED51B65, TXN_C485D16666]"
1,CUST_102615,1-Day Rapid Smurfing,2023-03-12 13:00:00,2023-03-13 13:00:00,2,87206.60,"[TXN_C485D16666, TXN_B8E3ABEEFE]"
2,CUST_103058,1-Day Rapid Smurfing,2023-05-29 15:00:00,2023-05-30 15:00:00,2,72442.71,"[TXN_69C3B979AA, TXN_46962F0527]"
3,CUST_103265,1-Day Rapid Smurfing,2023-03-06 12:00:00,2023-03-07 12:00:00,2,62108.55,"[TXN_C8BD1907A2, TXN_ECD0A88442]"
4,CUST_103265,1-Day Rapid Smurfing,2023-03-08 14:00:00,2023-03-09 14:00:00,2,68471.48,"[TXN_589B9CA56A, TXN_47308B03CA]"


Only 28 incidents where they attempted strucuting within the same day, some people call this type of structuring ATM hopping. This is almost always a clear structuring activity.

Scenario 2 - & day Standard Structuring

In [5]:
scenario2_alerts = []

for cust_id, group in cash.groupby('customer_id'):
    group = group.sort_values('timestamp').reset_index(drop=True)
    
    for i in range(len(group)):
        window_start = group.loc[i, 'timestamp']
        window_end = window_start + pd.Timedelta(days=7)
        
        window_txns = group[(group['timestamp'] >= window_start) & (group['timestamp'] < window_end)]
        
        count = len(window_txns)
        total = window_txns['amount_aed_equivalent'].sum()
        
        if count >= 2 and total >= 50000:
            scenario2_alerts.append({
                'customer_id': cust_id,
                'scenario': '7-Day Standard Structuring',
                'window_start': window_start,
                'window_end': window_end,
                'num_deposits': count,
                'total_amount': round(total, 2),
                'transaction_ids': list(window_txns['transaction_id'])
            })

scenario2_df = pd.DataFrame(scenario2_alerts)
scenario2_df = scenario2_df.drop_duplicates(subset=['customer_id', 'window_start'])

print(f"Scenario 2 alerts: {len(scenario2_df)}")
scenario2_df.head()

Scenario 2 alerts: 256


,customer_id,scenario,window_start,window_end,num_deposits,total_amount,transaction_ids
0,CUST_100239,7-Day Standard Structuring,2023-06-29 13:00:00,2023-07-06 13:00:00,4,53956.69,"[TXN_92DB34CDF6, TXN_5554855B9C, TXN_FC65E46D2..."
1,CUST_100302,7-Day Standard Structuring,2023-08-30 10:00:00,2023-09-06 10:00:00,4,68829.41,"[TXN_6CB4E64AC0, TXN_D415C011CB, TXN_F40538B0B..."
2,CUST_100302,7-Day Standard Structuring,2023-08-31 14:00:00,2023-09-07 14:00:00,3,50992.23,"[TXN_D415C011CB, TXN_F40538B0B7, TXN_57433CBB10]"
3,CUST_100330,7-Day Standard Structuring,2023-07-29 10:00:00,2023-08-05 10:00:00,4,53355.73,"[TXN_89DA997838, TXN_3B09EAD3CF, TXN_502077574..."
4,CUST_100603,7-Day Standard Structuring,2023-06-14 10:00:00,2023-06-21 10:00:00,4,77252.29,"[TXN_09109069B7, TXN_653DA7F397, TXN_B376AA24A..."


The 7-day rolling window produced **256 alerts**, which is much higher than the **28 alerts** from the 1-day window. This is expected because the longer window can capture customers who spread their deposits over several days instead of making them all within a single day. These patterns would not be picked up by the 1-day rule but become visible when looking at the customer's activity over a 7-day period.


In [6]:
scenario3_alerts = []

for cust_id, group in cash.groupby('customer_id'):
    group = group.sort_values('timestamp').reset_index(drop=True)
    
    for i in range(len(group)):
        window_start = group.loc[i, 'timestamp']
        window_end = window_start + pd.Timedelta(days=30)
        
        window_txns = group[(group['timestamp'] >= window_start) & (group['timestamp'] < window_end)]
        
        count = len(window_txns)
        total = window_txns['amount_aed_equivalent'].sum()
        
        if count >= 8 and total >= 150000:
            scenario3_alerts.append({
                'customer_id': cust_id,
                'scenario': '30-Day Sustained Micro-Structuring',
                'window_start': window_start,
                'window_end': window_end,
                'num_deposits': count,
                'total_amount': round(total, 2),
                'transaction_ids': list(window_txns['transaction_id'])
            })

scenario3_df = pd.DataFrame(scenario3_alerts)
scenario3_df = scenario3_df.drop_duplicates(subset=['customer_id', 'window_start'])

print(f"Scenario 3 alerts: {len(scenario3_df)}")
scenario3_df.head()

Scenario 3 alerts: 0


""


Scenario 3 alerts are zero because the way the data is generated, the generated limited structuring scenarios at 7 deposits, and this logic rule we applied needed 8+ deposits, so it has never triggered any alert, this is not a logic error but a database limitation. The caveat I was talking about.

Let's now move onto **Validation**

In [7]:
struct_scenario_customers = (
    ground_truth[ground_truth['scenario_type'] == 'Structuring']
    .merge(accounts[['account_id', 'customer_id']], on='account_id')
    ['customer_id'].unique()
)

print(f"Customers with true structuring scenarios (ground truth): {len(struct_scenario_customers)}")

s1_flagged = set(scenario1_df['customer_id'].unique())
s2_flagged = set(scenario2_df['customer_id'].unique())
any_flagged = s1_flagged | s2_flagged

true_positives = any_flagged & set(struct_scenario_customers)
false_positives = any_flagged - set(struct_scenario_customers)
false_negatives = set(struct_scenario_customers) - any_flagged

print(f"\nFlagged by Scenario 1 or 2: {len(any_flagged)}")
print(f"True positives (correctly flagged): {len(true_positives)}")
print(f"False positives (flagged but not in ground truth): {len(false_positives)}")
print(f"False negatives (missed): {len(false_negatives)}")

precision = len(true_positives) / len(any_flagged) if any_flagged else 0
recall = len(true_positives) / len(struct_scenario_customers) if struct_scenario_customers.size else 0

print(f"\nPrecision: {precision:.1%}")
print(f"Recall: {recall:.1%}")

Customers with true structuring scenarios (ground truth): 149

Flagged by Scenario 1 or 2: 140
True positives (correctly flagged): 140
False positives (flagged but not in ground truth): 0
False negatives (missed): 9

Precision: 100.0%
Recall: 94.0%


Recall (94%) tells us how many of the actual structuring customers we managed to catch. Out of 149 customers involved in structuring, the model flagged 140 and missed 9. This is a strong result and shows that using both the 1-day and 7-day rolling windows helps capture most structuring patterns, including both rapid deposits and deposits spread across several days.

Precision (100%) tells us how many of the customers we flagged were actually involved in structuring. In this case, every customer flagged was a true structuring case, with no false positives. However, this needs to be interpreted carefully. In this synthetic dataset, cash deposits only exist as part of the injected structuring scenarios. There are no normal customers making legitimate cash deposits, such as small businesses, customers who regularly deal with cash, or other legitimate cash-heavy activities. Because this normal cash activity is missing from the dataset, the rule has nothing legitimate to potentially confuse with structuring. So the 100% precision mainly reflects the limitations of the dataset rather than proving that the rule would achieve the same result in a real bank. A real production system would need a much wider range of legitimate cash behaviour to properly test false positives and precision.

The 9 missed customers are probably the more useful finding here. The structuring scenarios were generated with around 4–7 deposits spread across different dates, so some customers may have had their deposits spread slightly beyond the 7-day window, for example over 8–10 days. These customers would not meet either the 1-day or 7-day rule. This shows that there is a genuine coverage gap in the current detection logic. In a real implementation, this could be addressed by adding another window, such as 10 or 14 days, or by adjusting the 30-day scenario so its minimum deposit count is low enough to capture shorter but more spread-out structuring patterns.

## Rapid Movement (Velocity + Time correlation) 

This scenario looks for potential funnel or pass through account behaviour, where a customer receives an unusually large credit and then moves most of it out again within a short period. This could indicate that the account is being used mainly to temporarily move funds rather than for normal banking activity.

Instead of using a fixed AED amount, I define a large inflow as a transaction that is at least 3 times the customer's median individual credit transaction. This makes the rule relative to the customer's own normal behaviour and avoids automatically flagging corporate customers simply because they regularly handle larger amounts.

Once a qualifying inflow is identified, I look at the next 24 hours. At least 95 percent of the inflow must leave the account to trigger a domestic pass through alert. If any of the outflow goes to a high risk or offshore jurisdiction, the threshold is lowered to 80 percent because rapid movement towards these destinations carries additional risk.

These thresholds are internal detection rules rather than fixed regulatory requirements and would need further tuning using real transaction data and alert volumes.

Finally, the outflow must be spread across 3 or fewer counterparties. If the money is split across more recipients, it starts to look more like a layering pattern, which is handled separately to avoid overlap between the detection scenarios.

In [8]:
# building median credit for each customer
credits_all = transactions[transactions['direction'] == 'Credit'].copy()

median_credit_by_cust = (
    credits_all.groupby('customer_id')['amount_aed_equivalent']
    .median()
    .rename('median_credit_size')
)

count    1.000000e+04
mean     6.523795e+04
std      1.055572e+05
min      1.371755e+03
25%      1.270275e+04
50%      3.006138e+04
75%      7.109992e+04
max      1.331218e+06
Name: median_credit_size, dtype: float64

In [9]:
#flag qualifying large inflows per customer
credits_all = credits_all.merge(median_credit_by_cust, on='customer_id', how='left')
credits_all['is_large_inflow'] = credits_all['amount_aed_equivalent'] >= (3 * credits_all['median_credit_size'])

large_inflows = credits_all[credits_all['is_large_inflow']].copy()
large_inflows['timestamp'] = pd.to_datetime(large_inflows['timestamp'])

print(f"Total large inflows flagged: {len(large_inflows)}")
print(f"Unique customers with at least one large inflow: {large_inflows['customer_id'].nunique()}")
large_inflows[['customer_id', 'timestamp', 'amount_aed_equivalent', 'median_credit_size']].head()

Total large inflows flagged: 1235
Unique customers with at least one large inflow: 418


,customer_id,timestamp,amount_aed_equivalent,median_credit_size
308,CUST_100016,2023-03-30,1371022.00,3759.00
1400,CUST_100073,2023-03-09,1541535.55,153279.13
2249,CUST_100108,2023-07-11,633699.53,60871.69
2282,CUST_100110,2023-03-01,1072735.60,68591.28
3174,CUST_100162,2023-02-04,1473478.00,19233.00


Note that these numbers for example, are derived from judgement and not based on any laws or standard practices. 3 times of median should cover most of the regular transaction for most people and it is taken to balance the detection of rapid movement of funds and false positive. The best way to get the absolute number is to reiterate the experiment with different numbers and find which one produces the best result, but because the data is synthetic, it will definatly create some overfitting problems and won't generalise well. 

Now we want to know how much of a debit that has happend to this large inflow within 24 hours (again this can be 48, there is no right answer here). I have given different countries different sensitivity in detection, high risk countries and transfer to those countries will trigger an alert even if the outflow propotion is comparitively lower, for example it will be triggered even if the customer send 80% of inflow money to a high risk country, similar to an offshore country.

In [10]:
debits_all = transactions[transactions['direction'] == 'Debit'].copy()
debits_all['timestamp'] = pd.to_datetime(debits_all['timestamp'])

high_risk_countries = ['Iran', 'Syria', 'North Korea', 'Myanmar']
offshore_countries = ['Cayman Islands', 'BVI', 'Panama', 'Seychelles']
watch_countries = high_risk_countries + offshore_countries

rapid_movement_alerts = []

for idx, row in large_inflows.iterrows():
    cust_id = row['customer_id']
    inflow_amt = row['amount_aed_equivalent']
    window_start = row['timestamp']
    window_end = window_start + pd.Timedelta(hours=24)
    
    cust_debits = debits_all[
        (debits_all['customer_id'] == cust_id) &
        (debits_all['timestamp'] >= window_start) &
        (debits_all['timestamp'] < window_end)
    ]
    
    if cust_debits.empty:
        continue
    
    total_outflow = cust_debits['amount_aed_equivalent'].sum()
    outflow_ratio = total_outflow / inflow_amt
    num_counterparties = cust_debits['counterparty_id'].nunique()
    
    touches_watch_country = cust_debits['counterparty_country'].isin(watch_countries).any()
    
    threshold = 0.80 if touches_watch_country else 0.95
    
    if outflow_ratio >= threshold and num_counterparties <= 3:
        rapid_movement_alerts.append({
            'customer_id': cust_id,
            'inflow_timestamp': window_start,
            'inflow_amount': round(inflow_amt, 2),
            'total_outflow': round(total_outflow, 2),
            'outflow_ratio': round(outflow_ratio, 4),
            'num_counterparties': num_counterparties,
            'touches_high_risk_or_offshore': touches_watch_country,
            'threshold_applied': threshold,
            'transaction_ids': list(cust_debits['transaction_id'])
        })

rapid_movement_df = pd.DataFrame(rapid_movement_alerts)
print(f"Rapid Movement alerts: {len(rapid_movement_df)}")
rapid_movement_df.head()

Rapid Movement alerts: 61


,customer_id,inflow_timestamp,inflow_amount,total_outflow,outflow_ratio,num_counterparties,touches_high_risk_or_offshore,threshold_applied,transaction_ids
0,CUST_100251,2023-03-04 10:00:00,173396.00,163073.10,0.9405,3,True,0.8,"[TXN_AEC884AD43, TXN_2187FD77AC, TXN_40BCAB8305]"
1,CUST_100311,2023-08-24 11:00:00,654307.29,635758.85,0.9717,3,True,0.8,"[TXN_740DF961F5, TXN_AC43A6125B, TXN_1A7CD80116]"
2,CUST_100412,2023-10-07 10:00:00,217268.00,207180.20,0.9536,2,True,0.8,"[TXN_1BE64C461B, TXN_B2E4AC59D8]"
3,CUST_100459,2023-03-24 11:00:00,227878.00,220535.05,0.9678,1,True,0.8,[TXN_1172205037]
4,CUST_100528,2023-01-22 09:00:00,115389.00,109342.39,0.9476,1,True,0.8,[TXN_69E03F6BD7]


Out of 1,235 unusually large inflows across 418 customers, only 61 alerts were raised. This is expected because most large transactions are not suspicious on their own. A customer may receive a large but legitimate payment, such as a bonus, asset sale or inheritance, without immediately moving most of the money out again. The rule only flags cases where the large inflow is followed by rapid and near-total outflow within 24 hours.


**Validation of Rapid Movement of funds**

In [13]:
rapid_scenario_customers = (
    ground_truth[ground_truth['scenario_type'] == 'Rapid Movement']
    .merge(accounts[['account_id', 'customer_id']], on='account_id')
    ['customer_id'].unique()
)

print(f"Customers with true Rapid Movement scenarios (ground truth): {len(rapid_scenario_customers)}")

flagged = set(rapid_movement_df['customer_id'].unique())
true_pos = flagged & set(rapid_scenario_customers)
false_pos = flagged - set(rapid_scenario_customers)
false_neg = set(rapid_scenario_customers) - flagged

precision = len(true_pos) / len(flagged) if flagged else 0
recall = len(true_pos) / len(rapid_scenario_customers) if len(rapid_scenario_customers) else 0

print(f"Flagged: {len(flagged)}")
print(f"True positives: {len(true_pos)}")
print(f"False positives: {len(false_pos)}")
print(f"False negatives: {len(false_neg)}")
print(f"\nPrecision: {precision:.1%}")
print(f"Recall: {recall:.1%}")

Customers with true Rapid Movement scenarios (ground truth): 99
Flagged: 61
True positives: 59
False positives: 2
False negatives: 40

Precision: 96.7%
Recall: 59.6%


Precision (96.7%) is strong. Out of the 61 customers flagged, 59 were actually involved in the injected Rapid Movement scenarios, with only 2 false positives. This suggests that the combination of a large relative inflow, a high outflow ratio, a short time window and a maximum of 3 counterparties gives a fairly reliable signal when the rule is triggered.

Recall (59.6%) is lower. Out of 99 actual scenarios, the rule detected 59 and missed 40. The main reason is likely the way a large inflow is defined. Using 3 times the customer's median credit transaction is a reasonable starting point, but it may miss suspicious transactions for customers who already normally receive large credits. Some missed cases may also have moved funds to more than 3 counterparties, making them more suitable for the separate layering scenario.

I am leaving the thresholds as they are rather than continuously adjusting them to improve the results against these specific synthetic scenarios. The purpose here is to build a reasonable and explainable detection rule based on the customer's own transaction history, destination risk and transaction behaviour. In a real production environment, these results would be used as a starting point for further tuning based on the balance between detecting more suspicious activity and keeping the number of alerts manageable for investigators.

## Part 3 - Layering

Objective: Detect potential layering where funds move through a short chain of intermediary accounts before reaching a final external destination, making the transaction trail more difficult to trace.

Core conditions: Account A sends funds to Account B, B forwards a similar amount to Account C, and C sends it to an external counterparty, with a minimum of 3 hops from A to B to C to external, each hop must happen within 72 hours of the previous one, each account must forward between 85 percent and 100 percent of the amount received to allow for small differences such as fees or rounding, and the final destination can be domestic or offshore, although higher risk destinations should increase alert severity rather than being required for the pattern.

Exclusions: Payroll, dividend and tax related transactions should be excluded from chain building where payment type information is available, as these are generally expected recurring payment types and could otherwise create false positives.

Note: One to many rapid dispersion is a separate layering pattern where a single inflow is split across multiple counterparties, but this is not included here because the synthetic dataset scenarios follow a linear multi hop structure.

**Step 1 — isolate transfer-type transactions between accounts (not merchant/card spending)**

In [16]:
transfers = transactions[
    (transactions['transaction_type'] == 'Transfer')
].copy()
transfers['timestamp'] = pd.to_datetime(transfers['timestamp'])

print(f"Total transfer transactions: {len(transfers)}")
transfers[['transaction_id', 'account_id', 'timestamp', 'amount_aed_equivalent', 'direction', 'counterparty_id']].head()

Total transfer transactions: 1078211


,transaction_id,account_id,timestamp,amount_aed_equivalent,direction,counterparty_id
2,TXN_78A2828EA9,ACC_1000000,2023-01-12 18:00:00,33.40,Debit,CP_10140
5,TXN_5D2D948F35,ACC_1000000,2023-01-19 21:00:00,2165.87,Debit,CP_10140
10,TXN_276965FF27,ACC_1000000,2023-02-09 18:00:00,846.23,Debit,CP_11063
22,TXN_84484325D4,ACC_1000000,2023-03-27 11:00:00,3062.07,Debit,CP_10140
24,TXN_25E166A77A,ACC_1000000,2023-03-29 17:00:00,853.69,Debit,CP_10140


Step 2 - matching debit-credit pairs into hops.

In [17]:
internal_transfers = transfers[transfers['counterparty_id'].astype(str).str.startswith('ACC_')].copy()

print(f"Internal account-to-account transfers: {len(internal_transfers)}")
internal_transfers[['transaction_id', 'account_id', 'timestamp', 'amount_aed_equivalent', 'direction', 'counterparty_id']].head(10)

Internal account-to-account transfers: 200


,transaction_id,account_id,timestamp,amount_aed_equivalent,direction,counterparty_id
60440,TXN_799BD0C777,ACC_1000128,2023-07-11,633699.53,Credit,ACC_1003151
61104,TXN_154F6E3F74,ACC_1000131,2023-03-01,1072735.60,Credit,ACC_1006165
83548,TXN_1903FF7A79,ACC_1000186,2023-01-12,413497.79,Debit,ACC_1008325
104375,TXN_E4F0CF2C87,ACC_1000229,2023-09-17,459459.00,Credit,ACC_1005869
104377,TXN_91F7D014E2,ACC_1000229,2023-09-19,440389.17,Debit,ACC_1005786
181488,TXN_AD8AE44A89,ACC_1000382,2023-10-18,271964.00,Credit,ACC_1005101
181490,TXN_9D8544973E,ACC_1000382,2023-10-20,248390.78,Debit,ACC_1006801
219768,TXN_76EAC5F943,ACC_1000446,2023-07-25,240335.00,Debit,ACC_1003302
251274,TXN_5E2F4F9A9B,ACC_1000515,2023-07-08,437478.00,Debit,ACC_1011517
272245,TXN_3DAE11D086,ACC_1000555,2023-10-05,680225.55,Credit,ACC_1008105


Step 3 - build the graph

In [18]:
import networkx as nx

G = nx.MultiDiGraph()

for _, row in internal_transfers[internal_transfers['direction'] == 'Debit'].iterrows():
    G.add_edge(
        row['account_id'],
        row['counterparty_id'],
        transaction_id=row['transaction_id'],
        timestamp=row['timestamp'],
        amount=row['amount_aed_equivalent']
    )

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 149
Edges: 100


In [19]:
from itertools import product

layering_chains = []

for start_node in G.nodes():
    for _, mid_node, edge1 in G.out_edges(start_node, data=True):
        for _, end_node, edge2 in G.out_edges(mid_node, data=True):
            if end_node == start_node:
                continue
            
            t1 = pd.to_datetime(edge1['timestamp'])
            t2 = pd.to_datetime(edge2['timestamp'])
            
            if t2 < t1 or (t2 - t1) > pd.Timedelta(hours=72):
                continue
            
            amt1 = edge1['amount']
            amt2 = edge2['amount']
            decay_ratio = amt2 / amt1
            
            if 0.85 <= decay_ratio <= 1.0:
                layering_chains.append({
                    'hop1_from': start_node,
                    'hop1_to': mid_node,
                    'hop1_txn': edge1['transaction_id'],
                    'hop1_amount': round(amt1, 2),
                    'hop1_timestamp': t1,
                    'hop2_from': mid_node,
                    'hop2_to': end_node,
                    'hop2_txn': edge2['transaction_id'],
                    'hop2_amount': round(amt2, 2),
                    'hop2_timestamp': t2,
                    'decay_ratio': round(decay_ratio, 4)
                })

layering_df = pd.DataFrame(layering_chains)
print(f"2-hop chains found: {len(layering_df)}")
layering_df.head()

2-hop chains found: 35


,hop1_from,hop1_to,hop1_txn,hop1_amount,hop1_timestamp,hop2_from,hop2_to,hop2_txn,hop2_amount,hop2_timestamp,decay_ratio
0,ACC_1000186,ACC_1008325,TXN_1903FF7A79,413497.79,2023-01-12,ACC_1008325,ACC_1001105,TXN_3419A0BD22,407834.30,2023-01-14,0.9863
1,ACC_1000446,ACC_1003302,TXN_76EAC5F943,240335.00,2023-07-25,ACC_1003302,ACC_1010958,TXN_CB1F95EA7B,229942.89,2023-07-26,0.9568
2,ACC_1000515,ACC_1011517,TXN_5E2F4F9A9B,437478.00,2023-07-08,ACC_1011517,ACC_1001001,TXN_AFB301DDD1,413348.57,2023-07-10,0.9448
3,ACC_1000776,ACC_1011144,TXN_58288C8A06,402114.00,2023-01-17,ACC_1011144,ACC_1006681,TXN_C010575606,383209.27,2023-01-18,0.9530
4,ACC_1000978,ACC_1003678,TXN_B5115AA65B,388745.00,2023-10-18,ACC_1003678,ACC_1005124,TXN_25BB828207,371466.13,2023-10-20,0.9556


In [20]:
scenario_customers_layering = set(
    ground_truth[ground_truth['scenario_type'] == 'Layering']['scenario_id']
)
print(f"Total Layering scenarios in ground truth: {len(scenario_customers_layering)}")

detected_hop1_txns = set(layering_df['hop1_txn'])
gt_hop1_candidates = ground_truth[
    (ground_truth['scenario_type'] == 'Layering') &
    (ground_truth['scenario_role'] == 'layer_source')
]

missed = gt_hop1_candidates[~gt_hop1_candidates['transaction_id'].isin(detected_hop1_txns)]
print(f"Missed hop-1 transactions: {len(missed)}")
missed_scenario_ids = missed['scenario_id'].unique()

missed_full = ground_truth[ground_truth['scenario_id'].isin(missed_scenario_ids)].merge(
    transactions[['transaction_id','timestamp','amount_aed_equivalent']], on='transaction_id'
).sort_values(['scenario_id','timestamp'])

missed_full

Total Layering scenarios in ground truth: 50
Missed hop-1 transactions: 15


,transaction_id,account_id,scenario_id,scenario_type,scenario_role,is_suspicious,timestamp,amount_aed_equivalent
30,TXN_171BB38415,ACC_1009204,SCEN_LAY_1df0f7,Layering,layer_source,True,2023-07-08,815280.31
31,TXN_3E1C38450A,ACC_1001465,SCEN_LAY_1df0f7,Layering,layer_hop_1,True,2023-07-08,221996.00
32,TXN_6FE2ECBD76,ACC_1001465,SCEN_LAY_1df0f7,Layering,layer_hop_1,True,2023-07-10,204271.68
33,TXN_B5ACB62F99,ACC_1002228,SCEN_LAY_1df0f7,Layering,layer_hop_2,True,2023-07-10,204271.68
34,TXN_7D96A2F9FD,ACC_1002228,SCEN_LAY_1df0f7,Layering,layer_destination,True,2023-07-11,189130.65
...,...,...,...,...,...,...,...,...
5,TXN_71B9B3D274,ACC_1001708,SCEN_LAY_f8c09e,Layering,layer_source,True,2023-01-20,1728263.81
6,TXN_6604408ABC,ACC_1005998,SCEN_LAY_f8c09e,Layering,layer_hop_1,True,2023-01-20,470596.00
7,TXN_F1793C241A,ACC_1005998,SCEN_LAY_f8c09e,Layering,layer_hop_1,True,2023-01-21,453158.11
8,TXN_F0C33EA12C,ACC_1001502,SCEN_LAY_f8c09e,Layering,layer_hop_2,True,2023-01-21,1664223.16


In [22]:
missed_summary = []

for s_id in missed_scenario_ids:
    scen_txns = ground_truth[ground_truth['scenario_id'] == s_id].merge(
        transactions[['transaction_id','timestamp','amount_aed_equivalent']], on='transaction_id'
    ).sort_values('timestamp')
    
    source = scen_txns[scen_txns['scenario_role'] == 'layer_source']
    hop1_out = scen_txns[(scen_txns['scenario_role'] == 'layer_hop_1')].sort_values('timestamp')
    
    if len(source) == 0 or len(hop1_out) < 2:
        continue
    
    t1 = source['timestamp'].iloc[0]
    amt1 = source['amount_aed_equivalent'].iloc[0]
    
    hop1_debit = hop1_out.iloc[-1]
    t2 = hop1_debit['timestamp']
    amt2 = hop1_debit['amount_aed_equivalent']
    
    hours_gap = (t2 - t1).total_seconds() / 3600
    decay = amt2 / amt1 if amt1 else None
    
    missed_summary.append({
        'scenario_id': s_id,
        'hop1_amount': round(amt1, 2),
        'hop2_amount': round(amt2, 2),
        'hours_gap': round(hours_gap, 1),
        'decay_ratio': round(decay, 4) if decay else None
    })

pd.DataFrame(missed_summary)

,scenario_id,hop1_amount,hop2_amount,hours_gap,decay_ratio
0,SCEN_LAY_b96db9,590094.45,146253.30,24.0,0.2478
1,SCEN_LAY_f8c09e,1728263.81,453158.11,24.0,0.2622
2,SCEN_LAY_d7ab3b,524987.55,137077.85,24.0,0.2611
3,SCEN_LAY_9549f3,140706.00,515843.65,24.0,3.6661
4,SCEN_LAY_effaa7,218227.00,743993.96,24.0,3.4093
5,SCEN_LAY_bcd8c3,738701.34,190261.36,24.0,0.2576
6,SCEN_LAY_1df0f7,815280.31,204271.68,48.0,0.2506
7,SCEN_LAY_74ca29,757692.95,179700.33,24.0,0.2372
8,SCEN_LAY_e3f6e5,182836.00,642622.94,24.0,3.5148
9,SCEN_LAY_299f4d,455119.00,1540011.46,24.0,3.3838


In [24]:
check = ground_truth[ground_truth['scenario_id'] == 'SCEN_LAY_b96db9'].drop(columns=['account_id']).merge(
    transactions[['transaction_id','account_id','timestamp','amount','currency','amount_aed_equivalent']],
    on='transaction_id'
).sort_values('timestamp')

check[['scenario_role','account_id','timestamp','amount','currency','amount_aed_equivalent']]

,scenario_role,account_id,timestamp,amount,currency,amount_aed_equivalent
0,layer_source,ACC_1003094,2023-02-06,149391.00,EUR,590094.45
1,layer_hop_1,ACC_1007693,2023-02-06,149391.00,AED,149391.00
2,layer_hop_1,ACC_1007693,2023-02-07,146253.30,AED,146253.30
3,layer_hop_2,ACC_1001157,2023-02-07,146253.30,AED,146253.30
4,layer_destination,ACC_1001157,2023-02-08,135511.58,AED,135511.58


Investigation of the 15 missed layering chains showed that the issue was a synthetic data generation artifact rather than a problem with the detection logic. The generator carried forward the same raw amount across each hop without converting it using the currency and FX rate of the next account. So when a chain passed through EUR or USD accounts, the amount_aed_equivalent was correctly calculated for that transaction, but the next hop was generated using the original foreign currency amount, creating an artificial 3.7 to 4x jump or drop in the AED normalized ratio. This was confirmed by checking the missed cases, where the ratios clustered around 0.25 or 3.5 to 3.7, matching the EUR and USD FX rates almost exactly. For example, a EUR 149391 transaction had an AED equivalent of 590094, but the next hop was recorded as AED 149391, meaning the same raw number was carried forward instead of being converted. All 15 missed cases showed the same pattern, so I am treating this as a known limitation of the synthetic data rather than changing the data or detection logic after the fact. In a real dataset, the decay ratio should always be calculated using a properly currency normalized amount at each hop, not the raw transaction amount, which is exactly the issue this edge case ended up highlighting.


Step 4 - Checking the third hop

In [25]:
external_transfers = transfers[transfers['counterparty_id'].astype(str).str.startswith('CP_')].copy()
external_transfers = external_transfers.merge(
    counterparties[['cp_id', 'country', 'cp_type']],
    left_on='counterparty_id', right_on='cp_id', how='left'
)

full_chains = []

for _, chain in layering_df.iterrows():
    end_account = chain['hop2_to']
    hop1_time = chain['hop1_timestamp']
    hop2_amount = chain['hop2_amount']
    
    window_end = hop1_time + pd.Timedelta(hours=72)
    
    candidate_hop3 = external_transfers[
        (external_transfers['account_id'] == end_account) &
        (external_transfers['direction'] == 'Debit') &
        (external_transfers['timestamp'] >= chain['hop2_timestamp']) &
        (external_transfers['timestamp'] <= window_end)
    ]
    
    if candidate_hop3.empty:
        continue
    
    for _, hop3 in candidate_hop3.iterrows():
        decay3 = hop3['amount_aed_equivalent'] / hop2_amount
        if 0.85 <= decay3 <= 1.0:
            full_chains.append({
                **chain.to_dict(),
                'hop3_txn': hop3['transaction_id'],
                'hop3_to_country': hop3['country'],
                'hop3_amount': round(hop3['amount_aed_equivalent'], 2),
                'hop3_timestamp': hop3['timestamp'],
                'hop3_decay_ratio': round(decay3, 4)
            })

full_chains_df = pd.DataFrame(full_chains)
print(f"Full 3-hop layering chains found: {len(full_chains_df)}")
full_chains_df[['hop1_from','hop1_to','hop2_to','hop3_to_country','hop1_amount','hop2_amount','hop3_amount']].head()

Full 3-hop layering chains found: 21


,hop1_from,hop1_to,hop2_to,hop3_to_country,hop1_amount,hop2_amount,hop3_amount
0,ACC_1000446,ACC_1003302,ACC_1010958,BVI,240335.0,229942.89,220101.46
1,ACC_1000776,ACC_1011144,ACC_1006681,Seychelles,402114.0,383209.27,353548.08
2,ACC_1000978,ACC_1003678,ACC_1005124,Seychelles,388745.0,371466.13,356553.72
3,ACC_1001154,ACC_1004649,ACC_1005029,Cayman Islands,446039.0,420570.90,401016.58
4,ACC_1001506,ACC_1002236,ACC_1011792,Panama,241467.0,232594.15,222301.04


In [26]:
layering_scenario_ids = ground_truth[ground_truth['scenario_type'] == 'Layering']['scenario_id'].unique()

hop1_to_scenario = ground_truth[
    (ground_truth['scenario_type'] == 'Layering') &
    (ground_truth['scenario_role'] == 'layer_source')
][['transaction_id', 'scenario_id']].rename(columns={'transaction_id': 'hop1_txn'})

full_chains_df = full_chains_df.merge(hop1_to_scenario, on='hop1_txn', how='left')

detected_scenarios = set(full_chains_df['scenario_id'].dropna())
true_scenarios = set(layering_scenario_ids)

true_pos = detected_scenarios & true_scenarios
false_pos = detected_scenarios - true_scenarios
false_neg = true_scenarios - detected_scenarios

precision = len(true_pos) / len(detected_scenarios) if detected_scenarios else 0
recall = len(true_pos) / len(true_scenarios) if true_scenarios else 0

print(f"True layering scenarios: {len(true_scenarios)}")
print(f"Detected: {len(detected_scenarios)}")
print(f"True positives: {len(true_pos)}")
print(f"False positives: {len(false_pos)}")
print(f"False negatives: {len(false_neg)}")
print(f"\nPrecision: {precision:.1%}")
print(f"Recall: {recall:.1%}")

True layering scenarios: 50
Detected: 21
True positives: 21
False positives: 0
False negatives: 29

Precision: 100.0%
Recall: 42.0%


Validation results for the layering chain detection showed 100 percent precision, with all 21 detected 3 hop chains being genuine injected Layering scenarios and no false positives. I think this is a strong result because the combination of the A to B to C to external chain structure, 72 hour timing and 85 to 100 percent amount matching at each hop creates a very specific pattern that is unlikely to happen by chance in normal transactions. Recall was 42 percent, with 21 out of 50 true layering chains detected and 29 missed. The main reason for the missed cases is the currency conversion issue identified earlier. 15 scenarios failed the hop 1 to hop 2 decay check because non AED accounts carried forward the raw foreign currency amount instead of its AED equivalent. The remaining missed cases are likely to have the same issue at the second hop, since a single non AED account anywhere in the chain can cause the decay ratio to fail. I am keeping the current thresholds rather than loosening them to improve recall because that would mean tuning the rule around a known synthetic data issue rather than the actual AML pattern. So while the 42 percent recall is not ideal, I think the 100 percent precision and the confirmed root cause behind the missed cases make the result more meaningful and defensible than artificially increasing recall by weakening the detection logic.


In [27]:
#Lets consolidate the result I have got so far
summary = pd.DataFrame([
    {'Scenario': 'Structuring (1-day + 7-day)', 'True Cases': 149, 'Detected': 140, 'Precision': '100.0%', 'Recall': '94.0%'},
    {'Scenario': 'Rapid Movement', 'True Cases': 99, 'Detected': 61, 'Precision': '96.7%', 'Recall': '59.6%'},
    {'Scenario': 'Layering (3-hop)', 'True Cases': 50, 'Detected': 21, 'Precision': '100.0%', 'Recall': '42.0%'},
])
summary

,Scenario,True Cases,Detected,Precision,Recall
0,Structuring (1-day + 7-day),149,140,100.0%,94.0%
1,Rapid Movement,99,61,96.7%,59.6%
2,Layering (3-hop),50,21,100.0%,42.0%


In [28]:
#Now I'm going to do something interesting. A network graph visualisation for layering chain. 
!pip install pyvis

   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 756.0/756.0 kB 4.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [jsonpickle]
   -------------------- ------------------- 1/2 [pyvis]
   -------------------- ------------------- 1/2 [pyvis]
   ---------------------------------------- 2/2 [pyvis]



In [31]:
from pyvis.network import Network

net = Network(height='750px', width='100%', directed=True, notebook=True, cdn_resources='in_line')
net.barnes_hut(gravity=-8000, central_gravity=0.1, spring_length=250, spring_strength=0.02)

for _, chain in full_chains_df.iterrows():
    src = chain['hop1_from']
    mid = chain['hop1_to']
    end = chain['hop2_to']
    dest_country = chain['hop3_to_country']
    dest_label = f"{dest_country} (Offshore)"

    net.add_node(src, label=src.replace('ACC_', ''), color='#4CAF50', title=f'Source Account: {src}', size=18)
    net.add_node(mid, label=mid.replace('ACC_', ''), color='#FFC107', title=f'Intermediary Hop: {mid}', size=18)
    net.add_node(end, label=end.replace('ACC_', ''), color='#FF9800', title=f'Final Domestic Hop: {end}', size=18)
    net.add_node(dest_label, label=dest_country, color='#F44336', title='Offshore Destination', size=28, shape='diamond')

    net.add_edge(src, mid, value=chain['hop1_amount'], color='#999999',
                 title=f"AED {chain['hop1_amount']:,.0f} on {chain['hop1_timestamp'].date()}")
    net.add_edge(mid, end, value=chain['hop2_amount'], color='#999999',
                 title=f"AED {chain['hop2_amount']:,.0f} on {chain['hop2_timestamp'].date()}")
    net.add_edge(end, dest_label, value=chain['hop3_amount'], color='#999999',
                 title=f"AED {chain['hop3_amount']:,.0f} to {dest_country} on {chain['hop3_timestamp'].date()}")

net.set_options("""
{
  "physics": {
    "stabilization": { "iterations": 300 }
  }
}
""")

html_content = net.generate_html()

legend_html = """
<div style="position:absolute; top:10px; left:10px; background:white; padding:12px 16px;
            border:1px solid #ccc; border-radius:8px; font-family:Arial; font-size:13px;
            box-shadow: 2px 2px 6px rgba(0,0,0,0.15); z-index:1000;">
  <b>AML Layering Chain: A &rarr; B &rarr; C &rarr; Offshore</b><br><br>
  <span style="color:#4CAF50;">&#9679;</span> Source Account (Hop 1 origin)<br>
  <span style="color:#FFC107;">&#9679;</span> Intermediary Account (Hop 1 &rarr; Hop 2)<br>
  <span style="color:#FF9800;">&#9679;</span> Final Domestic Account (Hop 2 &rarr; Hop 3)<br>
  <span style="color:#F44336;">&#9670;</span> Offshore Destination Country<br><br>
  <i>Hover over any node or line for transaction details</i>
</div>
"""

html_content = html_content.replace('<body>', f'<body>{legend_html}')

with open('layering_network.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("Saved successfully")

from IPython.display import IFrame
IFrame('layering_network.html', width='100%', height=750)

Saved successfully


## Phase 3 Summary

Three deterministic detection scenarios were built and validated against `ground_truth.csv`:

| Scenario | True Cases | Detected | Precision | Recall |
|---|---|---|---|---|
| Structuring (1-day + 7-day) | 149 | 140 | 100.0% | 94.0% |
| Rapid Movement | 99 | 61 | 96.7% | 59.6% |
| Layering (3-hop) | 50 | 21 | 100.0% | 42.0% |

All three rules were designed from AML typology reasoning before any exposure to ground
truth, with ground truth used exclusively for post-hoc validation. Two genuine data
limitations were identified and documented during validation rather than silently tuned
around: the 30-day structuring tier could not be exercised due to the generator's injection
logic capping scenario length below its threshold, and roughly half of missed layering
chains trace to a currency-conversion artifact in how the generator carried amounts across
non-AED accounts. Both findings are discussed in detail in their respective sections above,
and reflect a deliberate choice to preserve methodological honesty over inflating headline
metrics.

False Positive Suppression logic (Scenario D) remains to be built next, applying contextual
overrides to reduce alert volume on legitimate high-value activity such as loan-funded
property purchases.